Importing Libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

train = pd.read_csv('train.csv')
public_test = pd.read_csv('public_test.csv')
private_test = pd.read_csv('private_test.csv')

print(train.shape, public_test.shape, private_test.shape)
train.head()

(10000, 14) (3000, 14) (3000, 13)


,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


Exploratory Data Analysis

In [3]:
print(train['Converted'].value_counts(normalize=True))
print()
print(train.isnull().sum())
print()
print(train.describe())

Converted
0    0.6913
1    0.3087
Name: proportion, dtype: float64

User_ID                  0
Age                   1480
Income                 984
City_Tier                0
Device_Type              0
Traffic_Source           0
Pages_Viewed             0
Products_Viewed          0
Time_On_Site          1848
Previous_Purchases       0
Discount_Seen            0
Browser_Version          0
Campaign_Code            0
Converted                0
dtype: int64

           User_ID          Age         Income     City_Tier  Pages_Viewed  \
count  10000.00000  8520.000000    9016.000000  10000.000000   10000.00000   
mean    5000.50000    41.457746   69961.772797      1.933400      15.60850   
std     2886.89568    13.770164   24790.673822      0.737712       8.62346   
min        1.00000    18.000000   12000.000000      1.000000       1.00000   
25%     2500.75000    29.000000   52294.644359      1.000000       8.00000   
50%     5000.50000    41.000000   70171.613672      2.000000      16.000

In [4]:
print(train['Device_Type'].unique())
print(train['Traffic_Source'].unique())
print(train.groupby('Device_Type')['Converted'].mean())
print(train.groupby('Traffic_Source')['Converted'].mean())

['Mobile' 'Desktop' 'Tablet']
['Organic' 'Social Media' 'Referral' 'Email' 'Paid Ads']
Device_Type
Desktop    0.330906
Mobile     0.300629
Tablet     0.306468
Name: Converted, dtype: float64
Traffic_Source
Email           0.349359
Organic         0.291762
Paid Ads        0.277123
Referral        0.381181
Social Media    0.303061
Name: Converted, dtype: float64


Feature Engineering

In [5]:
def feature_engineer(df):
    df = df.copy()
    # Engagement-rate features
    df['Pages_per_min'] = df['Pages_Viewed'] / (df['Time_On_Site'] + 1)
    df['Products_per_page'] = df['Products_Viewed'] / (df['Pages_Viewed'] + 1)
    df['Income_per_age'] = df['Income'] / (df['Age'] + 1)
    return df

train_fe = feature_engineer(train)
public_fe = feature_engineer(public_test)
private_fe = feature_engineer(private_test)

train_fe.head()

,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted,Pages_per_min,Products_per_page,Income_per_age
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0,0.471254,0.666667,1755.825573
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0,0.590446,0.250000,1350.063592
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0,0.070175,0.500000,1525.561435
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0,NaN,0.800000,1791.615762
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0,0.781250,1.400000,2923.034696


Preprocessing Pipeline

In [6]:
target = 'Converted'

num_cols = ['Age','Income','Pages_Viewed','Products_Viewed','Time_On_Site',
            'Previous_Purchases','City_Tier','Discount_Seen','Browser_Version','Campaign_Code',
            'Pages_per_min','Products_per_page','Income_per_age']
cat_cols = ['Device_Type','Traffic_Source']

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

Model Training & Validation

In [7]:
X = train_fe[num_cols + cat_cols]
y = train_fe[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight='balanced', random_state=42
    ))
])

model.fit(X_train, y_train)
val_pred = model.predict(X_val)
print('Validation F1:', f1_score(y_val, val_pred))
print(classification_report(y_val, val_pred))

Validation F1: 0.5672436750998668
              precision    recall  f1-score   support

           0       0.83      0.67      0.74      1383
           1       0.48      0.69      0.57       617

    accuracy                           0.68      2000
   macro avg       0.66      0.68      0.65      2000
weighted avg       0.72      0.68      0.69      2000



In [8]:
# Evaluate on public_test (acts as an additional hold-out)
X_pub = public_fe[num_cols + cat_cols]
y_pub = public_fe[target]
pub_pred = model.predict(X_pub)
print('Public Test F1:', f1_score(y_pub, pub_pred))

Public Test F1: 0.5200553250345782


Final Model (trained on train + public_test for maximum data)

In [9]:
combined = pd.concat([train_fe, public_fe], ignore_index=True)
X_final = combined[num_cols + cat_cols]
y_final = combined[target]

final_model = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight='balanced', random_state=42
    ))
])
final_model.fit(X_final, y_final)
print('Final model trained on', X_final.shape[0], 'records')

Final model trained on 13000 records


Predictions on private_test.csv


In [10]:
X_priv = private_fe[num_cols + cat_cols]
priv_pred = final_model.predict(X_priv)

submission = pd.DataFrame({
    'User_ID': private_test['User_ID'],
    'Converted': priv_pred.astype(int)
})

submission.to_csv('submission.csv', index=False)
print(submission['Converted'].value_counts())
submission.head()

Converted
0    1710
1    1290
Name: count, dtype: int64


,User_ID,Converted
0,103001,0
1,103002,1
2,103003,1
3,103004,1
4,103005,0
